# Mahie Spectrogram Generator

This notebook generates event-locked spectrogram figures and QC tables from precomputed LFP collection HDF5 files. The code is split into annotated sections so the pipeline can be reviewed and migrated into `src/coop_ephys_analysis/lfp/` more easily.


## Imports

Import standard-library utilities, numerical/scientific packages, plotting helpers, and notebook display support.


In [ ]:
from pathlib import Path
from types import SimpleNamespace
import re
import shutil
import warnings

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from matplotlib.cm import ScalarMappable
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm

try:
    from IPython.display import display
except ImportError:
    display = print

warnings.filterwarnings("ignore", category=RuntimeWarning)


## Paths

Define local input collection folders and the output folder for generated spectrogram figures and QC tables.


In [ ]:
DOWNLOADS = Path.home() / "Downloads"

COLLECTION_PATHS = {
    "day 1": DOWNLOADS / "D1_lfp_collection",
    "day 8": DOWNLOADS / "D8_lfp_collection",
}

OUTPUT_ROOT = DOWNLOADS / "7-10 spectrograms"

# Rebuild the output folder as though this code has never been run before.
RECREATE_OUTPUT_ROOT = True


## Analysis Settings

Set the metrics, behaviors, frequency ranges, regions, connections, event windows, resampling options, and global color-scale rules used by the full run.


In [ ]:
METRICS = (
    "power",
    "coherence",
    "granger",
)

BEHAVIORS = (
    "light",
    "nose poke",
)

FREQUENCY_RANGES = {
    "0-20 Hz": (0.0, 20.0),
    "20-100 Hz": (20.0, 100.0),
}

# Power remains regional.
POWER_REGIONS = (
    "mPFC",
    "IC",
    "NAc",
    "MD",
    "LH",
    "BLA",
)

# Nondirectional permitted coherence connections.
# Tuple format: (display label, first region, second region)
ACCEPTABLE_COHERENCE_CONNECTIONS = (
    ("mPFC–BLA", "mPFC", "BLA"),
    ("mPFC–IC", "mPFC", "IC"),
    ("mPFC–MD", "mPFC", "MD"),
    ("mPFC–NAc", "mPFC", "NAc"),
)

# Directed permitted Granger connections.
# IMPORTANT: tuple format is (display label, target region, source region).
# Therefore the HDF5 value is read as:
#     granger[..., target_index, source_index] = source -> target
ACCEPTABLE_GRANGER_CONNECTIONS = (
    ("BLA → mPFC", "mPFC", "BLA"),
    ("IC → mPFC", "mPFC", "IC"),
    ("MD → mPFC", "mPFC", "MD"),
    ("NAc → mPFC", "mPFC", "NAc"),
    ("mPFC → BLA", "BLA", "mPFC"),
    ("mPFC → IC", "IC", "mPFC"),
    ("mPFC → MD", "MD", "mPFC"),
    ("mPFC → NAc", "NAc", "mPFC"),
)

# All regions that must be available in every recording.
ANALYSIS_REGIONS = tuple(
    dict.fromkeys(
        list(POWER_REGIONS)
        + [region for _, first, second in ACCEPTABLE_COHERENCE_CONNECTIONS for region in (first, second)]
        + [region for _, target, source in ACCEPTABLE_GRANGER_CONNECTIONS for region in (target, source)]
    )
)

PRE_SECONDS = 2.0
POST_SECONDS = 2.0

# Canonical native spacing used to combine recordings.
TARGET_NATIVE_TIME_STEP = 0.5

# Requested display spacing.
DISPLAY_TIME_STEP = 0.2

# Use linear interpolation between native time estimates, followed by bicubic
# image rendering. This creates a noticeably stronger gradient than the prior
# nearest + bilinear version, while retaining a finite 0.2-second display grid
# so the result does not become an unstructured blur.
TIME_DISPLAY_RESAMPLING = "linear"
IMAGE_INTERPOLATION = "bicubic"
IMAGE_RESAMPLE = True

# Nose-poke filtering.
NOSE_POKE_LOCKOUT_SECONDS = 5.0
REQUIRE_REWARD_IF_REWARD_EVENTS_EXIST = True
REWARD_MATCH_MIN_SECONDS = 0.0
REWARD_MATCH_MAX_SECONDS = 4.0

# Granger transformation before session normalization.
GRANGER_TRANSFORM = "log1p"

# One balanced global scale per metric. The global median becomes the green
# midpoint. The 2nd and 98th percentiles define the blue and yellow ends, with
# separate lower and upper spans so skew on one side cannot flatten the other.
GLOBAL_LOWER_PERCENTILE = 2.0
GLOBAL_UPPER_PERCENTILE = 98.0

MINIMUM_HALF_SPAN = {
    "power": 0.05,
    "coherence": 0.05,
    "granger": 0.05,
}

FIGURE_DPI = 300
PANEL_WIDTH = 4.0
PANEL_HEIGHT = 2.45
EPSILON = 1e-12


## Recording Role Assignments

Map animal IDs onto analysis roles: subject, recipient, and subject-alone recordings.


In [ ]:
SUBJECT_IDS = {
    "1-2",
    "2-4",
    "4-4",
    "4-3",
}

RECIPIENT_IDS = {
    "1-3",
    "2-1",
    "4-2",
    "4-1",
}

ALONE_IDS = {
    "6-1",
    "6-3",
}


## Conditions

Define the behavioral conditions and condition order for light and nose-poke aligned analyses.


In [ ]:
CONDITIONS = {
    "selfish": {
        "role": "subject",
        "choice": "selfish",
        "title": "Selfish",
    },
    "prosocial": {
        "role": "subject",
        "choice": "prosocial",
        "title": "Prosocial",
    },
    "recipient_selfish": {
        "role": "recipient",
        "choice": "selfish",
        "title": "Recipient during selfish",
    },
    "recipient_prosocial": {
        "role": "recipient",
        "choice": "prosocial",
        "title": "Recipient during prosocial",
    },
    "selfish_alone": {
        "role": "subject_alone",
        "choice": "selfish",
        "title": "Selfish alone",
    },
    "prosocial_alone": {
        "role": "subject_alone",
        "choice": "prosocial",
        "title": "Prosocial alone",
    },
}

LIGHT_CONDITION_ORDER = (
    "selfish",
    "prosocial",
    "recipient_selfish",
    "recipient_prosocial",
    "selfish_alone",
    "prosocial_alone",
)

# Recipient conditions intentionally do not exist here.
NOSE_POKE_CONDITION_ORDER = (
    "selfish",
    "prosocial",
    "selfish_alone",
    "prosocial_alone",
)

CONDITION_ORDER_BY_BEHAVIOR = {
    "light": LIGHT_CONDITION_ORDER,
    "nose poke": NOSE_POKE_CONDITION_ORDER,
}


## Colormap

Create the shared blue-green-yellow colormap used for all saved spectrogram panels.


In [ ]:
# A continuous blue-green-yellow map. For each metric, the global median maps
# to green. Values below that median occupy the blue-to-green half, and values
# above it occupy the green-to-yellow half. Separate percentile spans preserve
# contrast on both sides even when the distribution is skewed.
BLUE_GREEN_YELLOW = LinearSegmentedColormap.from_list(
    "blue_green_yellow",
    [
        (0.00, "#08306B"),
        (0.25, "#2171B5"),
        (0.50, "#2CA25F"),
        (0.75, "#D9F0A3"),
        (1.00, "#FFF200"),
    ],
    N=256,
)
BLUE_GREEN_YELLOW.set_bad("#D9D9D9")
BLUE_GREEN_YELLOW.set_under("#08306B")
BLUE_GREEN_YELLOW.set_over("#FFF200")


## Configuration Validation

Fail early if the requested features, condition rules, Granger orientation, or display settings drift away from the intended analysis design.


In [ ]:
def validate_configuration():
    """Fail early if any requested restriction is configured incorrectly."""

    expected_coherence = {
        frozenset(("mPFC", "BLA")),
        frozenset(("mPFC", "IC")),
        frozenset(("mPFC", "MD")),
        frozenset(("mPFC", "NAc")),
    }
    configured_coherence = {
        frozenset((first, second))
        for _, first, second in ACCEPTABLE_COHERENCE_CONNECTIONS
    }
    assert configured_coherence == expected_coherence, (
        "The acceptable coherence list does not exactly match the requested four connections."
    )

    expected_granger = {
        ("mPFC", "BLA"),  # BLA -> mPFC: (target, source)
        ("mPFC", "IC"),
        ("mPFC", "MD"),
        ("mPFC", "NAc"),
        ("BLA", "mPFC"),
        ("IC", "mPFC"),
        ("MD", "mPFC"),
        ("NAc", "mPFC"),
    }
    configured_granger = {
        (target, source)
        for _, target, source in ACCEPTABLE_GRANGER_CONNECTIONS
    }
    assert configured_granger == expected_granger, (
        "The acceptable Granger list does not exactly match the requested eight directed connections."
    )

    assert not any(
        CONDITIONS[name]["role"] == "recipient"
        for name in NOSE_POKE_CONDITION_ORDER
    ), "Recipient conditions must not appear in nose-poke figures."

    assert len(set(NOSE_POKE_CONDITION_ORDER)) == 4
    assert len(ACCEPTABLE_COHERENCE_CONNECTIONS) == 4
    assert len(ACCEPTABLE_GRANGER_CONNECTIONS) == 8
    assert TIME_DISPLAY_RESAMPLING == "linear"
    assert IMAGE_INTERPOLATION == "bicubic"
    assert IMAGE_RESAMPLE is True
    assert 0.0 <= GLOBAL_LOWER_PERCENTILE < 50.0
    assert 50.0 < GLOBAL_UPPER_PERCENTILE <= 100.0
    assert GLOBAL_LOWER_PERCENTILE < GLOBAL_UPPER_PERCENTILE
    assert all(value > 0.0 for value in MINIMUM_HALF_SPAN.values())

    connection_labels = (
        [label for label, _, _ in ACCEPTABLE_COHERENCE_CONNECTIONS]
        + [label for label, _, _ in ACCEPTABLE_GRANGER_CONNECTIONS]
    )
    assert len(connection_labels) == len(set(connection_labels))

    print("Configuration checks passed.")


## Output Folder Setup

Prepare the output directory tree for metric/day/behavior-specific PNGs and summary CSVs.


In [ ]:
def recreate_output_directories():
    """Rebuild the output tree without touching either input collection."""

    if RECREATE_OUTPUT_ROOT and OUTPUT_ROOT.exists():
        shutil.rmtree(OUTPUT_ROOT)

    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

    for metric in METRICS:
        for day in COLLECTION_PATHS:
            for behavior in BEHAVIORS:
                (OUTPUT_ROOT / metric / day / behavior).mkdir(
                    parents=True,
                    exist_ok=True,
                )

    print("=" * 100)
    print("OUTPUT FOLDERS READY")
    print("=" * 100)
    print(OUTPUT_ROOT)


## Basic Hdf5 Helpers

Small utilities for decoding HDF5 values and reading mappings/events from collection files.


In [ ]:
def decode_h5_value(value):
    """Decode byte strings and NumPy scalar values."""

    if isinstance(value, bytes):
        return value.decode("utf-8")
    if isinstance(value, np.generic):
        return value.item()
    return value


def extract_animal_id(recording_name, subject=None):
    """Extract an animal identifier such as 1-2."""

    for token in str(recording_name).split("_"):
        match = re.fullmatch(r"(\d+)[.-](\d+)", token)
        if match:
            return f"{match.group(1)}-{match.group(2)}"

    subject_match = re.fullmatch(r"(\d+)[.-](\d+)", str(subject))
    if subject_match:
        return f"{subject_match.group(1)}-{subject_match.group(2)}"

    return None


def determine_role(animal_id):
    """Determine whether a recording is subject, recipient, alone, or unknown."""

    if animal_id in SUBJECT_IDS:
        return "subject"
    if animal_id in RECIPIENT_IDS:
        return "recipient"
    if animal_id in ALONE_IDS:
        return "subject_alone"
    return "unknown"


def read_integer_mapping(group):
    """Read an HDF5 integer mapping stored as attributes or scalar datasets."""

    mapping = {}
    if group is None:
        return mapping

    for key, value in group.attrs.items():
        try:
            mapping[str(key)] = int(decode_h5_value(value))
        except (TypeError, ValueError):
            pass

    for key, obj in group.items():
        if not isinstance(obj, h5py.Dataset):
            continue
        try:
            mapping[str(key)] = int(decode_h5_value(obj[()]))
        except (TypeError, ValueError):
            pass

    return mapping


def read_region_to_index(h5_file):
    """Read the actual region-to-array-index mapping."""

    mapping = read_integer_mapping(h5_file.get("brain region dict"))
    if not mapping:
        raise KeyError("Could not read the HDF5 'brain region dict'.")
    return mapping


def read_region_to_channel(h5_file):
    """Read the region-to-electrode-channel mapping when available."""

    return read_integer_mapping(h5_file.get("channels"))


def read_event_arrays(h5_file):
    """Read all event arrays from one HDF5 recording."""

    event_group = h5_file.get("event")
    if event_group is None:
        event_group = h5_file.get("events")
    if event_group is None:
        return {}

    events = {}
    for event_name, event_object in event_group.items():
        if isinstance(event_object, h5py.Dataset):
            events[str(event_name)] = np.asarray(event_object[()])
    return events


## Collection Metadata

Load recording-level metadata from the LFP collection without keeping large metric arrays in memory.


In [ ]:
def load_collection_metadata(day, collection_folder):
    """Read metadata without keeping every large spectral array in memory."""

    collection_folder = Path(collection_folder)

    if not collection_folder.exists():
        raise FileNotFoundError(f"Missing collection folder:\n{collection_folder}")

    collection_json = collection_folder / "lfp_collection.json"
    if not collection_json.exists():
        raise FileNotFoundError(f"Missing lfp_collection.json:\n{collection_json}")

    h5_files = sorted(collection_folder.rglob("*.h5"))
    if not h5_files:
        raise FileNotFoundError(f"No H5 files were found inside:\n{collection_folder}")

    recordings = []
    reference_frequencies = None

    for h5_path in h5_files:
        with h5py.File(h5_path, "r") as h5_file:
            if "data" not in h5_file:
                raise KeyError(f"{h5_path.name} has no data group.")

            data_group = h5_file["data"]
            if "frequencies" not in data_group:
                raise KeyError(f"{h5_path.name} has no data/frequencies dataset.")

            metadata = dict(h5_file["metadata"].attrs) if "metadata" in h5_file else {}

            recording_name = str(
                decode_h5_value(metadata.get("name", h5_path.stem))
            )
            subject = decode_h5_value(metadata.get("subject", None))

            time_step = float(
                decode_h5_value(
                    metadata.get(
                        "window step",
                        metadata.get("window_step", TARGET_NATIVE_TIME_STEP),
                    )
                )
            )
            window_duration = float(
                decode_h5_value(
                    metadata.get(
                        "window duration",
                        metadata.get("window_duration", 1.0),
                    )
                )
            )

            if time_step <= 0:
                raise ValueError(f"{h5_path.name} has an invalid window step: {time_step}")
            if window_duration <= 0:
                raise ValueError(
                    f"{h5_path.name} has an invalid window duration: {window_duration}"
                )

            frequencies = np.asarray(
                data_group["frequencies"][()]
            ).squeeze().astype(float)

            if frequencies.ndim != 1:
                raise ValueError(
                    f"{h5_path.name} frequencies must be one-dimensional; found {frequencies.shape}."
                )

            region_to_index = read_region_to_index(h5_file)
            region_to_channel = read_region_to_channel(h5_file)

            missing_regions = [
                region for region in ANALYSIS_REGIONS if region not in region_to_index
            ]
            if missing_regions:
                raise ValueError(
                    f"{h5_path.name} is missing expected regions.\n"
                    f"Missing: {missing_regions}\n"
                    f"Available: {sorted(region_to_index)}"
                )

            metric_shapes = {
                metric: tuple(data_group[metric].shape) if metric in data_group else None
                for metric in METRICS
            }
            events = read_event_arrays(h5_file)

        animal_id = extract_animal_id(recording_name, subject)

        recordings.append(
            SimpleNamespace(
                day=day,
                name=recording_name,
                path=h5_path,
                subject=subject,
                animal_id=animal_id,
                role=determine_role(animal_id),
                time_step=time_step,
                window_duration=window_duration,
                frequencies=frequencies,
                region_to_index=region_to_index,
                region_to_channel=region_to_channel,
                metric_shapes=metric_shapes,
                events=events,
            )
        )

        selected_frequencies = frequencies[
            (frequencies >= 0.0) & (frequencies <= 100.0)
        ]
        if selected_frequencies.size == 0:
            raise ValueError(f"{h5_path.name} contains no frequencies from 0 to 100 Hz.")

        if reference_frequencies is None:
            reference_frequencies = selected_frequencies.copy()

    return SimpleNamespace(
        day=day,
        folder=collection_folder,
        recordings=recordings,
        frequencies=reference_frequencies,
    )


def create_inventory_table(collections):
    """Create a quality-control table for all recordings."""

    rows = []
    for collection in collections.values():
        for recording in collection.recordings:
            rows.append(
                {
                    "day": recording.day,
                    "recording": recording.name,
                    "animal_id": recording.animal_id,
                    "role": recording.role,
                    "native_time_step_s": recording.time_step,
                    "window_duration_s": recording.window_duration,
                    "power_shape": recording.metric_shapes["power"],
                    "coherence_shape": recording.metric_shapes["coherence"],
                    "granger_shape": recording.metric_shapes["granger"],
                    "event_names": ", ".join(recording.events.keys()),
                }
            )
    return pd.DataFrame(rows)


def create_region_mapping_table(collections):
    """Save the actual HDF5 region index and electrode mapping."""

    rows = []
    for collection in collections.values():
        for recording in collection.recordings:
            for region, array_index in sorted(
                recording.region_to_index.items(),
                key=lambda item: item[1],
            ):
                rows.append(
                    {
                        "day": recording.day,
                        "recording": recording.name,
                        "animal_id": recording.animal_id,
                        "role": recording.role,
                        "region": region,
                        "array_index": array_index,
                        "original_channel": recording.region_to_channel.get(region, np.nan),
                    }
                )
    return pd.DataFrame(rows)


def create_feature_definition_table():
    """Document exactly which rows are used for each metric."""

    rows = []

    for region in POWER_REGIONS:
        rows.append(
            {
                "metric": "power",
                "display_label": region,
                "directionality": "regional",
                "matrix_target": "",
                "matrix_source": "",
                "matrix_read_rule": "power[..., region]",
            }
        )

    for label, first, second in ACCEPTABLE_COHERENCE_CONNECTIONS:
        rows.append(
            {
                "metric": "coherence",
                "display_label": label,
                "directionality": "nondirectional",
                "matrix_target": first,
                "matrix_source": second,
                "matrix_read_rule": (
                    f"mean(coherence[..., {first}, {second}], "
                    f"coherence[..., {second}, {first}])"
                ),
            }
        )

    for label, target, source in ACCEPTABLE_GRANGER_CONNECTIONS:
        rows.append(
            {
                "metric": "granger",
                "display_label": label,
                "directionality": "source -> target",
                "matrix_target": target,
                "matrix_source": source,
                "matrix_read_rule": f"granger[..., {target}, {source}]",
            }
        )

    return pd.DataFrame(rows)


## Event Helpers

Parse stored event names, find matching behavioral events, and convert event arrays into onset times in seconds.


In [ ]:
def choice_from_event_name(event_name):
    """Determine whether an event is selfish or prosocial."""

    name = str(event_name).lower()
    if "selfish" in name:
        return "selfish"
    if any(term in name for term in ("coop", "cooperative", "prosocial")):
        return "prosocial"
    return None


def behavior_from_event_name(event_name):
    """Determine the behavioral event category."""

    cleaned = re.sub(r"[^a-z0-9]+", "", str(event_name).lower())
    if "nosepoke" in cleaned:
        return "nose poke"
    if "reward" in cleaned:
        return "reward"
    if "light" in cleaned:
        return "light"
    return None


def find_matching_event(recording, choice, behavior):
    """Find the stored event matching one choice and behavior."""

    matching_events = [
        event_name
        for event_name in recording.events
        if choice_from_event_name(event_name) == choice
        and behavior_from_event_name(event_name) == behavior
    ]

    if not matching_events:
        return None

    return sorted(matching_events, key=lambda value: (len(str(value)), str(value)))[0]


def extract_event_onsets_seconds(recording, event_name, number_of_time_bins):
    """Extract event onsets and automatically convert milliseconds to seconds."""

    if event_name is None:
        return np.array([], dtype=float)

    values = np.asarray(recording.events[event_name], dtype=float)
    if values.size == 0:
        return np.array([], dtype=float)

    if values.ndim == 1:
        onsets = values
    elif values.ndim == 2 and values.shape[1] == 2:
        onsets = values[:, 0]
    elif values.ndim == 2 and values.shape[0] == 2:
        onsets = values[0, :]
    elif values.ndim == 2:
        onsets = values[:, 0]
    else:
        return np.array([], dtype=float)

    onsets = onsets[np.isfinite(onsets)]
    approximate_duration = number_of_time_bins * recording.time_step

    if onsets.size and np.nanmax(onsets) > 1.5 * approximate_duration:
        onsets = onsets / 1000.0

    return np.sort(onsets.astype(float))


def reward_matched_nose_pokes(nose_poke_onsets, reward_onsets):
    """Retain nose pokes followed by a reward in the configured interval."""

    nose_poke_onsets = np.asarray(nose_poke_onsets, dtype=float)
    reward_onsets = np.asarray(reward_onsets, dtype=float)

    if reward_onsets.size == 0:
        return nose_poke_onsets

    retained = []
    for nose_poke_time in nose_poke_onsets:
        delays = reward_onsets - nose_poke_time
        if np.any(
            (delays >= REWARD_MATCH_MIN_SECONDS)
            & (delays <= REWARD_MATCH_MAX_SECONDS)
        ):
            retained.append(nose_poke_time)

    return np.asarray(retained, dtype=float)


def filter_nose_pokes(recording, number_of_time_bins):
    """
    Reward-match each choice when matching reward events exist, combine choices,
    then keep only the first retained nose poke in every five-second lockout.
    """

    candidates = []

    for choice in ("selfish", "prosocial"):
        nose_poke_event = find_matching_event(recording, choice, "nose poke")
        if nose_poke_event is None:
            continue

        nose_poke_onsets = extract_event_onsets_seconds(
            recording,
            nose_poke_event,
            number_of_time_bins,
        )

        reward_event = find_matching_event(recording, choice, "reward")
        reward_onsets = extract_event_onsets_seconds(
            recording,
            reward_event,
            number_of_time_bins,
        )

        if (
            REQUIRE_REWARD_IF_REWARD_EVENTS_EXIST
            and reward_event is not None
            and reward_onsets.size > 0
        ):
            retained_onsets = reward_matched_nose_pokes(
                nose_poke_onsets,
                reward_onsets,
            )
        else:
            retained_onsets = nose_poke_onsets

        candidates.extend((float(onset), nose_poke_event) for onset in retained_onsets)

    candidates.sort(key=lambda item: item[0])

    retained_by_event = {}
    previous_retained_onset = -np.inf

    for onset, event_name in candidates:
        if onset - previous_retained_onset >= NOSE_POKE_LOCKOUT_SECONDS:
            retained_by_event.setdefault(event_name, []).append(onset)
            previous_retained_onset = onset

    return {
        event_name: np.asarray(onsets, dtype=float)
        for event_name, onsets in retained_by_event.items()
    }


## Axis Resampling

Execute this section of the spectrogram generation workflow.


In [ ]:
def nearest_resample_axis(array, old_axis, new_axis, axis):
    """Nearest-neighbor resampling along one array axis."""

    old_axis = np.asarray(old_axis, dtype=float)
    new_axis = np.asarray(new_axis, dtype=float)

    if old_axis.ndim != 1 or new_axis.ndim != 1:
        raise ValueError("old_axis and new_axis must both be one-dimensional.")
    if old_axis.size == 0:
        raise ValueError("old_axis cannot be empty.")

    nearest_indices = np.asarray(
        [int(np.nanargmin(np.abs(old_axis - new_value))) for new_value in new_axis],
        dtype=int,
    )
    return np.take(array, nearest_indices, axis=axis)


def native_relative_time_axis():
    """Canonical native time points between -2 and +2 seconds."""

    return np.arange(
        -PRE_SECONDS,
        POST_SECONDS + TARGET_NATIVE_TIME_STEP / 2.0,
        TARGET_NATIVE_TIME_STEP,
    )


def display_relative_time_axis():
    """Requested 0.2-second display grid."""

    return np.arange(
        -PRE_SECONDS,
        POST_SECONDS + DISPLAY_TIME_STEP / 2.0,
        DISPLAY_TIME_STEP,
    )


def resample_panel_for_display(panel, original_times, new_times):
    """
    Linearly interpolate each frequency row onto the display-time grid.

    The numerical analysis remains on the native spectral estimates. This
    interpolation is used only to render a smoother visual gradient. Missing
    values are ignored when at least two finite samples remain in a row.
    """

    if TIME_DISPLAY_RESAMPLING != "linear":
        raise ValueError("TIME_DISPLAY_RESAMPLING must be 'linear'.")

    panel = np.asarray(panel, dtype=float)
    original_times = np.asarray(original_times, dtype=float)
    new_times = np.asarray(new_times, dtype=float)

    if panel.ndim != 2:
        raise ValueError("panel must have shape [frequency, time].")
    if original_times.ndim != 1 or new_times.ndim != 1:
        raise ValueError("original_times and new_times must be one-dimensional.")
    if panel.shape[1] != original_times.size:
        raise ValueError(
            "The panel time dimension must equal the length of original_times."
        )
    if original_times.size == 0:
        raise ValueError("original_times cannot be empty.")
    if np.any(np.diff(original_times) <= 0):
        raise ValueError("original_times must be strictly increasing.")

    interpolated = np.full(
        (panel.shape[0], new_times.size),
        np.nan,
        dtype=float,
    )

    for frequency_index, row in enumerate(panel):
        finite = np.isfinite(row) & np.isfinite(original_times)
        number_finite = int(np.sum(finite))

        if number_finite >= 2:
            interpolated[frequency_index] = np.interp(
                new_times,
                original_times[finite],
                row[finite],
                left=np.nan,
                right=np.nan,
            )
        elif number_finite == 1:
            only_time = float(original_times[finite][0])
            only_value = float(row[finite][0])
            nearest_index = int(np.nanargmin(np.abs(new_times - only_time)))
            interpolated[frequency_index, nearest_index] = only_value

    return interpolated


## Load One Metric

Execute this section of the spectrogram generation workflow.


In [ ]:
def load_recording_metric(recording, metric, reference_frequencies):
    """
    Load one metric from one recording and restrict region axes to
    ANALYSIS_REGIONS.

    Expected shapes:
        power:      [time, frequency, region]
        coherence:  [time, frequency, region, region]
        granger:    [time, frequency, target, source]
    """

    with h5py.File(recording.path, "r") as h5_file:
        data_group = h5_file["data"]
        if metric not in data_group:
            return None

        dataset = data_group[metric]
        expected_dimensions = 3 if metric == "power" else 4

        if dataset.ndim != expected_dimensions:
            raise ValueError(
                f"{recording.name} | {metric}\n"
                f"Expected {expected_dimensions} dimensions, found shape {dataset.shape}."
            )

        if dataset.shape[1] != len(recording.frequencies):
            raise ValueError(
                f"{recording.name} | {metric}\n"
                "Frequency does not appear to be axis 1."
            )

        frequency_indices = np.flatnonzero(
            (recording.frequencies >= 0.0) & (recording.frequencies <= 100.0)
        )
        if frequency_indices.size == 0:
            raise ValueError(f"No 0-100 Hz frequencies were found in {recording.name}.")

        frequency_start = int(frequency_indices[0])
        frequency_stop = int(frequency_indices[-1]) + 1

        region_indices = [
            recording.region_to_index[region] for region in ANALYSIS_REGIONS
        ]

        if metric == "power":
            if max(region_indices) >= dataset.shape[2]:
                raise IndexError(
                    f"{recording.name} power region mapping exceeds array shape {dataset.shape}."
                )
            # Read the contiguous block first, then reorder in NumPy.
            # This avoids h5py fancy-index ordering restrictions when the
            # HDF5 region indices are not already sorted.
            metric_array = np.asarray(
                dataset[:, frequency_start:frequency_stop, :],
                dtype=float,
            )
            metric_array = metric_array[:, :, region_indices]
        else:
            if max(region_indices) >= dataset.shape[2] or max(region_indices) >= dataset.shape[3]:
                raise IndexError(
                    f"{recording.name} {metric} region mapping exceeds array shape {dataset.shape}."
                )
            metric_array = np.asarray(
                dataset[:, frequency_start:frequency_stop, :, :],
                dtype=float,
            )
            metric_array = metric_array[:, :, region_indices, :]
            metric_array = metric_array[:, :, :, region_indices]

    local_frequencies = recording.frequencies[frequency_start:frequency_stop]

    if (
        len(local_frequencies) != len(reference_frequencies)
        or not np.allclose(local_frequencies, reference_frequencies)
    ):
        metric_array = nearest_resample_axis(
            array=metric_array,
            old_axis=local_frequencies,
            new_axis=reference_frequencies,
            axis=1,
        )

    metric_array[~np.isfinite(metric_array)] = np.nan
    return metric_array


## Feature Extraction and Session Normalization

Execute this section of the spectrogram generation workflow.


In [ ]:
def robust_session_zscore(values):
    """
    Normalize each frequency/feature against its full-session median:
        z = (value - median) / (1.4826 * MAD)
    Standard deviation is used only where MAD is invalid or approximately zero.
    """

    values = np.asarray(values, dtype=float)

    session_median = np.nanmedian(values, axis=0)
    median_absolute_deviation = np.nanmedian(
        np.abs(values - session_median[None, ...]),
        axis=0,
    )
    robust_scale = 1.4826 * median_absolute_deviation
    standard_deviation = np.nanstd(values, axis=0)

    invalid_scale = ~np.isfinite(robust_scale) | (robust_scale < EPSILON)
    robust_scale = np.where(invalid_scale, standard_deviation, robust_scale)

    invalid_scale = ~np.isfinite(robust_scale) | (robust_scale < EPSILON)
    robust_scale = np.where(invalid_scale, np.nan, robust_scale)

    return (values - session_median[None, ...]) / robust_scale[None, ...]


def analysis_region_index(region):
    """Return a region's index after the array is restricted to ANALYSIS_REGIONS."""

    return ANALYSIS_REGIONS.index(region)


def extract_raw_metric_features(metric_array, metric):
    """
    Convert a loaded metric into only its permitted displayed features.

    Returns:
        power:      [time, frequency, 6 regions]
        coherence:  [time, frequency, 4 nondirectional permitted edges]
        granger:    [time, frequency, 8 directed permitted edges]
    """

    metric_array = np.asarray(metric_array, dtype=float)

    if metric == "power":
        power_indices = [analysis_region_index(region) for region in POWER_REGIONS]
        return metric_array[:, :, power_indices]

    if metric == "coherence":
        edges = []
        for _, first_region, second_region in ACCEPTABLE_COHERENCE_CONNECTIONS:
            first_index = analysis_region_index(first_region)
            second_index = analysis_region_index(second_region)

            first_to_second = metric_array[:, :, first_index, second_index]
            second_to_first = metric_array[:, :, second_index, first_index]

            # Coherence is nondirectional. Average the two symmetric matrix cells.
            nondirectional_edge = np.nanmean(
                np.stack((first_to_second, second_to_first), axis=-1),
                axis=-1,
            )
            edges.append(nondirectional_edge)

        return np.stack(edges, axis=-1)

    if metric == "granger":
        edges = []
        for _, target_region, source_region in ACCEPTABLE_GRANGER_CONNECTIONS:
            target_index = analysis_region_index(target_region)
            source_index = analysis_region_index(source_region)

            # REQUIRED ORIENTATION:
            # granger[..., target, source] = source -> target
            directed_edge = metric_array[:, :, target_index, source_index]
            edges.append(directed_edge)

        return np.stack(edges, axis=-1)

    raise ValueError(f"Unsupported metric: {metric}")


def convert_metric_to_feature_values(metric_array, metric):
    """Transform and robustly normalize only the permitted displayed features."""

    raw_features = extract_raw_metric_features(metric_array, metric)

    if metric == "power":
        transformed = np.log10(np.maximum(raw_features, EPSILON))

    elif metric == "coherence":
        transformed = np.clip(raw_features, 0.0, 1.0)

    elif metric == "granger":
        transformed = np.maximum(raw_features, 0.0)

        if GRANGER_TRANSFORM == "log1p":
            transformed = np.log1p(transformed)
        elif GRANGER_TRANSFORM == "log10":
            transformed = np.log10(transformed + EPSILON)
        elif GRANGER_TRANSFORM == "none":
            pass
        else:
            raise ValueError(
                "GRANGER_TRANSFORM must be 'log1p', 'log10', or 'none'."
            )

    else:
        raise ValueError(f"Unsupported metric: {metric}")

    return robust_session_zscore(transformed)


def metric_feature_labels(metric):
    """Return the exact ordered row labels for a metric."""

    if metric == "power":
        return list(POWER_REGIONS)
    if metric == "coherence":
        return [label for label, _, _ in ACCEPTABLE_COHERENCE_CONNECTIONS]
    if metric == "granger":
        return [label for label, _, _ in ACCEPTABLE_GRANGER_CONNECTIONS]
    raise ValueError(f"Unsupported metric: {metric}")


## Synthetic Logic Tests

Execute this section of the spectrogram generation workflow.


In [ ]:
def run_internal_logic_tests():
    """Test edge selection, Granger orientation, resampling, and scale behavior."""

    number_of_regions = len(ANALYSIS_REGIONS)
    synthetic = np.zeros((3, 4, number_of_regions, number_of_regions), dtype=float)

    # Fill every directed matrix cell with a unique target/source code.
    for target in range(number_of_regions):
        for source in range(number_of_regions):
            synthetic[:, :, target, source] = 100.0 * target + source

    granger_features = extract_raw_metric_features(synthetic, "granger")
    assert granger_features.shape == (3, 4, 8)

    for feature_index, (_, target_region, source_region) in enumerate(
        ACCEPTABLE_GRANGER_CONNECTIONS
    ):
        target_index = analysis_region_index(target_region)
        source_index = analysis_region_index(source_region)
        expected = 100.0 * target_index + source_index
        assert np.all(granger_features[:, :, feature_index] == expected), (
            f"Granger orientation failed for {source_region} -> {target_region}."
        )

    coherence_features = extract_raw_metric_features(synthetic, "coherence")
    assert coherence_features.shape == (3, 4, 4)

    for feature_index, (_, first_region, second_region) in enumerate(
        ACCEPTABLE_COHERENCE_CONNECTIONS
    ):
        first_index = analysis_region_index(first_region)
        second_index = analysis_region_index(second_region)
        expected = np.mean(
            [
                100.0 * first_index + second_index,
                100.0 * second_index + first_index,
            ]
        )
        assert np.allclose(coherence_features[:, :, feature_index], expected)

    test_panel = np.array([[0.0, 1.0, 2.0]])
    resampled = resample_panel_for_display(
        test_panel,
        original_times=np.array([-1.0, 0.0, 1.0]),
        new_times=np.array([-1.0, -0.6, 0.0, 0.6, 1.0]),
    )
    expected_resampled = np.array([[0.0, 0.4, 1.0, 1.6, 2.0]])
    assert np.allclose(resampled, expected_resampled, equal_nan=True)

    test_values = np.array([-100.0, -2.0, -1.0, 0.0, 1.0, 3.0, 100.0])
    scale = calculate_balanced_scale_from_values(test_values, "power")
    assert np.isclose(scale["center"], 0.0)
    assert scale["vmin"] < scale["center"] < scale["vmax"]
    assert np.isclose(scale["fraction_below_center"], 3.0 / 7.0)
    assert np.isclose(scale["fraction_above_center"], 3.0 / 7.0)

    print("Internal logic tests passed.")


## Event-Aligned Session Averaging

Execute this section of the spectrogram generation workflow.


In [ ]:
def calculate_session_event_average(feature_values, event_onsets, recording):
    """
    Extract -2 to +2 second native windows, average events within a recording,
    then resample that session average to the common native time axis.
    """

    relative_times = np.arange(
        -PRE_SECONDS,
        POST_SECONDS + recording.time_step / 2.0,
        recording.time_step,
    )

    stored_time_centers = (
        recording.window_duration / 2.0
        + np.arange(feature_values.shape[0]) * recording.time_step
    )

    event_windows = []
    edge_skipped_count = 0

    for event_onset in np.asarray(event_onsets, dtype=float):
        desired_centers = float(event_onset) + relative_times

        if (
            desired_centers[0] < stored_time_centers[0]
            or desired_centers[-1] > stored_time_centers[-1]
        ):
            edge_skipped_count += 1
            continue

        right_indices = np.clip(
            np.searchsorted(stored_time_centers, desired_centers),
            0,
            len(stored_time_centers) - 1,
        )
        left_indices = np.clip(
            right_indices - 1,
            0,
            len(stored_time_centers) - 1,
        )

        use_left = (
            np.abs(stored_time_centers[left_indices] - desired_centers)
            <= np.abs(stored_time_centers[right_indices] - desired_centers)
        )
        selected_indices = np.where(use_left, left_indices, right_indices)

        mismatch = np.abs(stored_time_centers[selected_indices] - desired_centers)
        if np.any(mismatch > recording.time_step / 2.0 + 1e-9):
            edge_skipped_count += 1
            continue

        event_windows.append(feature_values[selected_indices])

    if not event_windows:
        return None, edge_skipped_count, 0

    session_average = np.nanmean(np.stack(event_windows, axis=0), axis=0)
    canonical_times = native_relative_time_axis()

    if (
        len(relative_times) != len(canonical_times)
        or not np.allclose(relative_times, canonical_times)
    ):
        session_average = nearest_resample_axis(
            array=session_average,
            old_axis=relative_times,
            new_axis=canonical_times,
            axis=0,
        )

    return session_average, edge_skipped_count, len(event_windows)


## Processing Accumulators

Execute this section of the spectrogram generation workflow.


In [ ]:
def initialize_accumulators():
    """Create only conditions permitted for each behavior."""

    return {
        behavior: {
            condition_name: {
                "session_averages": [],
                "recordings": [],
                "raw_events": 0,
                "kept_events": 0,
                "valid_windows": 0,
                "edge_skipped": 0,
            }
            for condition_name in CONDITION_ORDER_BY_BEHAVIOR[behavior]
        }
        for behavior in BEHAVIORS
    }


def process_metric_for_collection(collection, metric):
    """Process one metric while loading only one recording at a time."""

    accumulators = initialize_accumulators()

    print("\n" + "=" * 100)
    print(f"{collection.day.upper()} | {metric.upper()}")
    print("=" * 100)

    for recording_number, recording in enumerate(collection.recordings, start=1):
        if recording.role == "unknown":
            print(f"Skipping unknown role: {recording.name}")
            continue

        metric_array = load_recording_metric(
            recording=recording,
            metric=metric,
            reference_frequencies=collection.frequencies,
        )

        if metric_array is None:
            print(f"Missing {metric}: {recording.name}")
            continue

        feature_values = convert_metric_to_feature_values(metric_array, metric)
        number_of_time_bins = feature_values.shape[0]

        # Recipient recordings never enter nose-poke filtering or averaging.
        retained_nose_pokes = (
            filter_nose_pokes(recording, number_of_time_bins)
            if recording.role != "recipient"
            else {}
        )

        for behavior in BEHAVIORS:
            # Extra hard guard against recipient nose-poke processing.
            if behavior == "nose poke" and recording.role == "recipient":
                continue

            for condition_name in CONDITION_ORDER_BY_BEHAVIOR[behavior]:
                condition_info = CONDITIONS[condition_name]
                if condition_info["role"] != recording.role:
                    continue

                choice = condition_info["choice"]
                event_name = find_matching_event(recording, choice, behavior)
                if event_name is None:
                    continue

                raw_onsets = extract_event_onsets_seconds(
                    recording,
                    event_name,
                    number_of_time_bins,
                )

                if behavior == "nose poke":
                    kept_onsets = retained_nose_pokes.get(
                        event_name,
                        np.array([], dtype=float),
                    )
                else:
                    kept_onsets = raw_onsets

                result_slot = accumulators[behavior][condition_name]
                result_slot["raw_events"] += len(raw_onsets)
                result_slot["kept_events"] += len(kept_onsets)

                session_average, edge_skipped, valid_window_count = (
                    calculate_session_event_average(
                        feature_values=feature_values,
                        event_onsets=kept_onsets,
                        recording=recording,
                    )
                )

                result_slot["edge_skipped"] += edge_skipped
                result_slot["valid_windows"] += valid_window_count

                if session_average is not None:
                    result_slot["session_averages"].append(session_average)
                    result_slot["recordings"].append(recording.name)

        del metric_array
        del feature_values

        print(
            f"[{recording_number:02d}/{len(collection.recordings):02d}] "
            f"{recording.animal_id} | {recording.role}"
        )

    final_results = {behavior: {} for behavior in BEHAVIORS}

    for behavior in BEHAVIORS:
        for condition_name in CONDITION_ORDER_BY_BEHAVIOR[behavior]:
            result_slot = accumulators[behavior][condition_name]
            session_averages = result_slot.pop("session_averages")

            grand_average = (
                np.nanmean(np.stack(session_averages, axis=0), axis=0)
                if session_averages
                else None
            )

            final_results[behavior][condition_name] = {
                **result_slot,
                "grand_average": grand_average,
                "n_sessions": len(session_averages),
                "times": native_relative_time_axis(),
                "frequencies": collection.frequencies,
                "feature_labels": metric_feature_labels(metric),
            }

            print(
                f"{behavior:9s} | {condition_name:22s} | "
                f"sessions={len(session_averages)} | "
                f"raw={result_slot['raw_events']} | "
                f"kept={result_slot['kept_events']} | "
                f"valid={result_slot['valid_windows']} | "
                f"edge_skipped={result_slot['edge_skipped']}"
            )

    return final_results


## Global Balanced Color Scales

Compute one median-centered shared color scale per metric across all days, behaviors, and displayed conditions.


In [ ]:
def collect_metric_values(all_results, metric):
    """Collect every value that can actually appear in figures for one metric."""

    chunks = []

    for day in COLLECTION_PATHS:
        metric_results = all_results[day, metric]

        for behavior in BEHAVIORS:
            for condition_name in CONDITION_ORDER_BY_BEHAVIOR[behavior]:
                item = metric_results[behavior][condition_name]
                grand_average = item["grand_average"]

                if grand_average is None:
                    continue

                frequencies = np.asarray(item["frequencies"], dtype=float)
                frequency_mask = (frequencies >= 0.0) & (frequencies <= 100.0)
                values = grand_average[:, frequency_mask, :]
                finite_values = values[np.isfinite(values)]

                if finite_values.size:
                    chunks.append(finite_values)

    return np.concatenate(chunks) if chunks else np.array([], dtype=float)


def calculate_balanced_scale_from_values(values, metric):
    """
    Calculate one balanced shared color scale for a metric.

    The global median is mapped to green. Because the median divides the
    distribution, approximately half of the finite values fall on the
    blue-to-green side and half on the green-to-yellow side. The lower and
    upper scale spans are calculated independently from robust percentiles,
    preventing a long tail on one side from compressing the other side.
    """

    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    if values.size == 0:
        return {
            "vmin": -1.0,
            "center": 0.0,
            "vmax": 1.0,
            "number_of_values": 0,
            "fraction_below_center": np.nan,
            "fraction_above_center": np.nan,
        }

    center = float(np.nanmedian(values))
    lower_quantile = float(
        np.nanpercentile(values, GLOBAL_LOWER_PERCENTILE)
    )
    upper_quantile = float(
        np.nanpercentile(values, GLOBAL_UPPER_PERCENTILE)
    )

    lower_half_span = max(
        center - lower_quantile,
        MINIMUM_HALF_SPAN[metric],
    )
    upper_half_span = max(
        upper_quantile - center,
        MINIMUM_HALF_SPAN[metric],
    )

    vmin = center - lower_half_span
    vmax = center + upper_half_span

    if not np.isfinite(vmin) or not np.isfinite(center) or not np.isfinite(vmax):
        vmin, center, vmax = -1.0, 0.0, 1.0

    if not vmin < center:
        vmin = center - MINIMUM_HALF_SPAN[metric]
    if not center < vmax:
        vmax = center + MINIMUM_HALF_SPAN[metric]

    return {
        "vmin": float(vmin),
        "center": float(center),
        "vmax": float(vmax),
        "number_of_values": int(values.size),
        "fraction_below_center": float(np.mean(values < center)),
        "fraction_above_center": float(np.mean(values > center)),
    }


def calculate_metric_scales(all_results):
    """Calculate exactly one shared median-balanced scale per metric."""

    scales = {}
    for metric in METRICS:
        values = collect_metric_values(all_results, metric)
        scales[metric] = calculate_balanced_scale_from_values(values, metric)
    return scales


## Panel Extraction

Pull one feature-by-condition panel from the grand averages for a requested frequency band.


In [ ]:
def extract_display_panel(item, feature_index, frequency_low, frequency_high):
    """Extract one frequency-by-time panel and resample it for gradient display."""

    grand_average = item["grand_average"]
    if grand_average is None:
        return None

    frequencies = np.asarray(item["frequencies"], dtype=float)
    frequency_mask = (frequencies >= frequency_low) & (frequencies <= frequency_high)

    if not np.any(frequency_mask):
        return None

    native_panel = grand_average[:, frequency_mask, feature_index].T
    displayed_panel = resample_panel_for_display(
        panel=native_panel,
        original_times=item["times"],
        new_times=display_relative_time_axis(),
    )

    return displayed_panel, frequencies[frequency_mask]


## Plot Labels

Centralize human-readable metric descriptions and colorbar labels.


In [ ]:
def metric_description(metric):
    """Describe exactly what each figure row represents."""

    if metric == "power":
        return "regional log10 power, robustly normalized within each session"

    if metric == "coherence":
        return (
            "only the four permitted nondirectional mPFC coherence connections, "
            "robustly normalized within each session"
        )

    return (
        "only the eight permitted directed Granger connections; "
        "matrix read as [target, source] = source → target; "
        f"{GRANGER_TRANSFORM} transformed and robustly normalized within each session"
    )


def colorbar_label(metric):
    """Create a metric-specific colorbar label."""

    if metric == "power":
        return "Session-normalized log10 power"
    if metric == "coherence":
        return "Session-normalized coherence"
    return "Session-normalized Granger value"


## Figure Creation

Assemble and save the final spectrogram grid for one metric, day, behavior, and frequency range.


In [ ]:
def save_spectrogram_figure(
    collection,
    metric,
    behavior,
    frequency_label,
    frequency_range,
    results,
    metric_scale,
):
    """Save one complete spectrogram grid using the shared metric scale."""

    frequency_low, frequency_high = frequency_range
    active_conditions = CONDITION_ORDER_BY_BEHAVIOR[behavior]
    feature_labels = metric_feature_labels(metric)

    number_of_rows = len(feature_labels)
    number_of_columns = len(active_conditions)

    vmin = metric_scale["vmin"]
    center = metric_scale["center"]
    vmax = metric_scale["vmax"]

    normalization = TwoSlopeNorm(
        vmin=vmin,
        vcenter=center,
        vmax=vmax,
    )

    output_directory = OUTPUT_ROOT / metric / collection.day / behavior
    output_directory.mkdir(parents=True, exist_ok=True)
    output_path = output_directory / f"{frequency_label.replace(' ', '_')}.png"

    figure_width = PANEL_WIDTH * number_of_columns
    figure_height = max(8.5, PANEL_HEIGHT * number_of_rows)

    figure, axes = plt.subplots(
        nrows=number_of_rows,
        ncols=number_of_columns,
        figsize=(figure_width, figure_height),
        sharex=True,
        sharey=True,
        constrained_layout=True,
        squeeze=False,
    )

    display_times = display_relative_time_axis()

    for feature_index, feature_label in enumerate(feature_labels):
        for condition_index, condition_name in enumerate(active_conditions):
            axis = axes[feature_index, condition_index]
            item = results[behavior][condition_name]

            panel_result = extract_display_panel(
                item=item,
                feature_index=feature_index,
                frequency_low=frequency_low,
                frequency_high=frequency_high,
            )

            if panel_result is None:
                axis.set_facecolor("#E6E6E6")
                axis.text(
                    0.5,
                    0.5,
                    "NO DATA",
                    ha="center",
                    va="center",
                    transform=axis.transAxes,
                    fontsize=9,
                    fontweight="bold",
                )
            else:
                panel_values, selected_frequencies = panel_result

                # Saturate only values outside the globally selected percentiles.
                panel_values = np.clip(panel_values, vmin, vmax)

                axis.imshow(
                    panel_values,
                    origin="lower",
                    aspect="auto",
                    extent=[
                        display_times[0],
                        display_times[-1],
                        selected_frequencies[0],
                        selected_frequencies[-1],
                    ],
                    cmap=BLUE_GREEN_YELLOW,
                    norm=normalization,
                    interpolation=IMAGE_INTERPOLATION,
                    resample=IMAGE_RESAMPLE,
                )

            axis.axvline(
                0.0,
                color="black",
                linestyle="--",
                linewidth=0.8,
            )
            axis.set_xlim(-PRE_SECONDS, POST_SECONDS)
            axis.set_ylim(frequency_low, frequency_high)
            axis.set_xticks([-2, -1, 0, 1, 2])
            axis.tick_params(axis="both", labelsize=8)

            if feature_index == 0:
                axis.set_title(
                    f"{CONDITIONS[condition_name]['title']}\n"
                    f"n={item['n_sessions']}, events={item['valid_windows']}",
                    fontsize=10,
                    pad=8,
                )

            if condition_index == 0:
                axis.set_ylabel(
                    f"{feature_label}\nFrequency (Hz)",
                    fontsize=9,
                    labelpad=8,
                )

            if feature_index == number_of_rows - 1:
                axis.set_xlabel(
                    "Time from event (s)",
                    fontsize=9,
                    labelpad=5,
                )

    scalar_mappable = ScalarMappable(
        norm=normalization,
        cmap=BLUE_GREEN_YELLOW,
    )
    scalar_mappable.set_array([])

    colorbar = figure.colorbar(
        scalar_mappable,
        ax=axes.ravel().tolist(),
        shrink=0.92,
        pad=0.012,
        extend="both",
    )
    colorbar.set_label(colorbar_label(metric), fontsize=10)
    colorbar.set_ticks([vmin, center, vmax])
    colorbar.set_ticklabels(
        [
            f"{vmin:.2g}\nBlue",
            f"{center:.2g}\nGlobal median / green",
            f"{vmax:.2g}\nYellow",
        ]
    )
    colorbar.ax.tick_params(labelsize=9)

    number_of_panels = number_of_rows * number_of_columns

    figure.suptitle(
        f"{collection.day.title()} | {metric.title()} | "
        f"{behavior.title()} | {frequency_label}\n"
        f"{metric_description(metric)}\n"
        f"{number_of_panels} panels: {number_of_rows} rows × "
        f"{number_of_columns} conditions | one shared balanced {metric} scale: "
        f"{vmin:.2g} to {vmax:.2g}, green midpoint at global median "
        f"{center:.2g}\n"
        f"Native step: {TARGET_NATIVE_TIME_STEP:g} s | "
        f"display step: {DISPLAY_TIME_STEP:g} s | "
        f"time resampling: {TIME_DISPLAY_RESAMPLING} | "
        f"image interpolation: {IMAGE_INTERPOLATION}",
        fontsize=13,
        fontweight="bold",
    )

    figure.savefig(
        output_path,
        dpi=FIGURE_DPI,
        bbox_inches="tight",
        facecolor="white",
    )
    plt.close(figure)

    print("Saved:", output_path)
    return output_path


## Main Pipeline

Run the full workflow: validate settings, load metadata, process metrics, write QC tables, calculate scales, and save PNG figures.


In [ ]:
def main():
    validate_configuration()
    run_internal_logic_tests()
    recreate_output_directories()

    collections = {
        day: load_collection_metadata(day, collection_path)
        for day, collection_path in COLLECTION_PATHS.items()
    }

    inventory_df = create_inventory_table(collections)
    region_mapping_df = create_region_mapping_table(collections)
    feature_definition_df = create_feature_definition_table()

    inventory_df.to_csv(OUTPUT_ROOT / "recording_inventory.csv", index=False)
    region_mapping_df.to_csv(
        OUTPUT_ROOT / "region_index_and_channel_mapping.csv",
        index=False,
    )
    feature_definition_df.to_csv(
        OUTPUT_ROOT / "permitted_metric_rows.csv",
        index=False,
    )

    print("\n" + "=" * 100)
    print("RECORDING INVENTORY")
    print("=" * 100)
    display(inventory_df)

    print("\n" + "=" * 100)
    print("ACTUAL HDF5 REGION MAPPINGS")
    print("=" * 100)
    display(region_mapping_df)

    print("\n" + "=" * 100)
    print("PERMITTED FIGURE ROWS")
    print("=" * 100)
    display(feature_definition_df)

    all_results = {}
    for metric in METRICS:
        for day, collection in collections.items():
            all_results[day, metric] = process_metric_for_collection(
                collection,
                metric,
            )

    count_rows = []
    for metric in METRICS:
        for day in COLLECTION_PATHS:
            metric_results = all_results[day, metric]
            for behavior in BEHAVIORS:
                for condition_name in CONDITION_ORDER_BY_BEHAVIOR[behavior]:
                    item = metric_results[behavior][condition_name]
                    count_rows.append(
                        {
                            "day": day,
                            "metric": metric,
                            "behavior": behavior,
                            "condition": condition_name,
                            "condition_title": CONDITIONS[condition_name]["title"],
                            "n_sessions": item["n_sessions"],
                            "raw_events": item["raw_events"],
                            "kept_events": item["kept_events"],
                            "valid_windows": item["valid_windows"],
                            "edge_skipped": item["edge_skipped"],
                            "recipient_nose_poke_included": False
                            if behavior == "nose poke"
                            else "not applicable",
                        }
                    )

    event_count_df = pd.DataFrame(count_rows)
    event_count_df.to_csv(
        OUTPUT_ROOT / "event_processing_counts.csv",
        index=False,
    )

    print("\n" + "=" * 100)
    print("EVENT PROCESSING COUNTS")
    print("=" * 100)
    display(event_count_df)

    metric_scales = calculate_metric_scales(all_results)

    scale_rows = []
    for metric in METRICS:
        scale = metric_scales[metric]
        scale_rows.append(
            {
                "metric": metric,
                "display_vmin_blue": scale["vmin"],
                "display_center_global_median_green": scale["center"],
                "display_vmax_yellow": scale["vmax"],
                "lower_percentile_used": GLOBAL_LOWER_PERCENTILE,
                "upper_percentile_used": GLOBAL_UPPER_PERCENTILE,
                "fraction_values_below_green_center": (
                    scale["fraction_below_center"]
                ),
                "fraction_values_above_green_center": (
                    scale["fraction_above_center"]
                ),
                "number_of_values_used": scale["number_of_values"],
                "recipient_nose_poke_included": False,
                "shared_across": (
                    "D1, D8, light, nose poke, 0-20 Hz, 20-100 Hz, "
                    "all displayed conditions, permitted rows only"
                ),
            }
        )

    scale_df = pd.DataFrame(scale_rows)
    scale_df.to_csv(
        OUTPUT_ROOT / "balanced_global_color_scales.csv",
        index=False,
    )

    print("\n" + "=" * 100)
    print("ONE MEDIAN-BALANCED SHARED SCALE PER METRIC")
    print("=" * 100)
    display(scale_df)

    saved_files = []
    for metric in METRICS:
        shared_metric_scale = metric_scales[metric]

        for day, collection in collections.items():
            metric_results = all_results[day, metric]

            for behavior in BEHAVIORS:
                for frequency_label, frequency_range in FREQUENCY_RANGES.items():
                    saved_files.append(
                        save_spectrogram_figure(
                            collection=collection,
                            metric=metric,
                            behavior=behavior,
                            frequency_label=frequency_label,
                            frequency_range=frequency_range,
                            results=metric_results,
                            metric_scale=shared_metric_scale,
                        )
                    )

    print("\n" + "=" * 100)
    print("PIPELINE COMPLETE")
    print("=" * 100)
    print(f"Created {len(saved_files)} PNG files.")
    print("\nOutput directory:")
    print(OUTPUT_ROOT)

    print("\nLight columns:")
    for condition_name in LIGHT_CONDITION_ORDER:
        print("  " + CONDITIONS[condition_name]["title"])

    print("\nNose-poke columns — no recipient columns:")
    for condition_name in NOSE_POKE_CONDITION_ORDER:
        print("  " + CONDITIONS[condition_name]["title"])

    print("\nPower rows:")
    for label in metric_feature_labels("power"):
        print("  " + label)

    print("\nAcceptable coherence rows only:")
    for label in metric_feature_labels("coherence"):
        print("  " + label)

    print("\nAcceptable Granger rows only:")
    for label, target, source in ACCEPTABLE_GRANGER_CONNECTIONS:
        print(f"  {label}: [{target}, {source}] = {source} -> {target}")

    print("\nColor behavior:")
    print("  global median = green midpoint")
    print("  approximately half of values below the midpoint")
    print("  approximately half of values above the midpoint")
    print("  separate lower and upper percentile spans")

    print("\nRecipient nose-poke recordings were excluded from:")
    print("  processing")
    print("  event counts")
    print("  session averaging")
    print("  global scale calculation")
    print("  nose-poke figures")

    print("\nDisplay rendering:")
    print(
        f"  native {TARGET_NATIVE_TIME_STEP:g}-s values -> "
        f"{DISPLAY_TIME_STEP:g}-s display grid with linear interpolation"
    )
    print(
        "  imshow interpolation='bicubic' and resample=True "
        "for a stronger gradient while retaining the finite display grid"
    )


if __name__ == "__main__":
    main()
